# Using the Groq API in Google Colab

This notebook shows how to install the Groq Python SDK, provide an API key, make a short test request, and choose a different model. Requests are processed by Groq's hosted service; the notebook does not download a model to Colab.

Run the cells **from top to bottom**. A Groq account and API key are needed to run the live request. An internet connection and available account quota are also required.

## 1. Create an API key and add it to Colab

1. Sign in to the [Groq Console](https://console.groq.com/) and create an API key in the [API Keys area](https://console.groq.com/keys).
2. In Google Colab, open the **Secrets** panel using the key icon in the left sidebar.
3. Add a secret with the exact name **`GROQ_API_KEY`** and paste the API key as its value. Enable **notebook access** for the secret.
4. Run the installation and setup cells below. If you add the secret after running setup, rerun the setup cell.

**Security:** Keep the API key private. Do not place it in notebook source code, share it in screenshots, or include it in saved outputs. Outside Colab, the setup script can also read an environment variable named `GROQ_API_KEY`.

## 2. Install the Groq SDK

The next two cells create an installation script and run it in the current Colab environment. An installation message confirms that the Python package was installed; it does **not** confirm that the API key or service is working.

In [1]:
%%writefile h_groq_install.py
"""Install Groq SDK in the active Colab runtime (no model download)."""
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "groq>=0.25"])
print("Groq Python SDK installed.")


Writing h_groq_install.py


In [2]:
!python h_groq_install.py


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 1.1 MB/s eta 0:00:00
Groq Python SDK installed.


## 3. Connect with the saved key

The setup script first looks for the `GROQ_API_KEY` environment variable. If it is missing, it attempts to load the secret from Colab and creates `H_GROQ_CLIENT` using the Groq SDK.

If setup prints **`Groq client ready`**, a key was found and the client was created. This is a local setup check, **not yet a live API test**. If no key is found, return to the Secrets panel and rerun the setup cell.

In [3]:
%%writefile h_groq_setup.py
"""Part H1: Connect to Groq without writing your API key into a notebook.

Recommended on Colab: save GROQ_API_KEY under Secrets (key icon) and grant
notebook access. Or set it as an environment variable before executing.
"""
import os

if not os.environ.get("GROQ_API_KEY"):
    try:
        from google.colab import userdata
        secret = userdata.get("GROQ_API_KEY")
        if secret:
            os.environ["GROQ_API_KEY"] = secret
    except Exception as exc:
        print("Colab secret not available:", type(exc).__name__)

H_GROQ_CLIENT = None
if os.environ.get("GROQ_API_KEY"):
    from groq import Groq
    H_GROQ_CLIENT = Groq(api_key=os.environ["GROQ_API_KEY"])
    print("Groq client ready. API key is NOT displayed.")
else:
    print("No GROQ_API_KEY found. Set it in Colab Secrets, then rerun this cell.")
    print("Do NOT paste a secret key directly into the notebook source.")


Writing h_groq_setup.py


**Do not paste an API key directly into this notebook.** Use Colab Secrets named `GROQ_API_KEY` (or an environment variable). This section can be safely skipped when no key is configured.


In [4]:
%run -i h_groq_setup.py


Groq client ready. API key is NOT displayed.


## 4. Test a live request

Run the following cell after setup. It sends a short prompt to the chosen model through the Groq API.

- **Success:** A short generated response is printed, followed by a confirmation that the API request completed.
- **Missing key:** The test is skipped and the cell explains how to configure `GROQ_API_KEY`.
- **Request error:** Check the key and notebook access, the model ID, network access, and the account's current rate limits or quota.

This cell makes a real API request, which consumes quota and may be billable on your account. The result depends on the model; it does not need to match a specific sentence.

In [6]:
# To use another supported model, change the MODEL value and rerun this cell.
MODEL = "openai/gpt-oss-20b"

if H_GROQ_CLIENT is None:
    print("Test not run: set GROQ_API_KEY in Colab Secrets and rerun the setup cell.")
else:
    try:
        response = H_GROQ_CLIENT.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": "Explain the importance of fast language models"}
            ],
        )
        answer = response.choices[0].message.content
        if answer and answer.strip():
            print("Response:", answer.strip())
            print("Groq API test succeeded (model:", response.model + ").")
        else:
            print("The API request completed, but the response contained no text.")
    except Exception as exc:
        print("Groq API test failed:", type(exc).__name__, str(exc))
        print("Check the key, available model, quota/rate limits, and connection.")

Response: ### Why “Fast” Language Models Matter

| Domain | Why Speed Is Critical | Consequence of Slower Models |
|--------|-----------------------|------------------------------|
| **Real‑time dialogue** (chatbots, virtual assistants, gaming) | Users expect instant responses; latency directly hurts the perceived quality. | Delayed replies, user churn, reduced trust. |
| **Edge / on‑device inference** (mobile, IoT, wearables) | Limited compute, battery, and network bandwidth. | Cannot deploy powerful models; must fall back to cloud. |
| **Large‑scale deployment** (search, recommendation, enterprise services) | Hundreds of thousands of requests per second; each token costs compute. | Higher operational costs, longer queue times. |
| **Training & experimentation** | Researchers iterate on hyper‑parameters, architecture changes, or new data. | Prolonged training cycles → slower progress, higher carbon footprint. |
| **Safety & fairness** (real‑time moderation, content filtering) | Modera

## 5. Change the model

In the test cell above, replace the value of `MODEL` with another **model ID** from the [Groq supported models list](https://console.groq.com/docs/models). For example:

```python
MODEL = "openai/gpt-oss-20b"  # initial example
# or
MODEL = "openai/gpt-oss-120b"  # alternative example
```

Only one `MODEL = ...` assignment should be active. Rerun the test cell after changing it; the same `model=MODEL` argument will use the new model. Different models can have different capabilities, limits, and prices. Model availability may change or depend on your account.

## Account / costs

A **Groq API key and available service quota are required**. A free tier may be available, but **do not assume unlimited free use**. Live requests consume quota and may incur charges depending on your account. Model availability and limits change, so check the [Groq supported models](https://console.groq.com/docs/models) and [current rate limits](https://console.groq.com/docs/rate-limits) before running more extensive tests.

**Troubleshooting:** A `401` or authentication error usually indicates a missing or invalid key; a `404` or model-related error can indicate an unavailable model ID; a `429` indicates a rate limit has been reached. Verify the exact error message and your account settings before retrying repeatedly. See the [Groq quickstart](https://console.groq.com/docs/quickstart) for an official setup example.